# KuaiRec: EDA để hiểu dữ liệu và thiết kế feed gợi ý

Chạy từng ô từ trên xuống bằng **Shift + Enter**. Sau mỗi bảng/biểu đồ, đọc câu hỏi rồi thử sửa mã.
Notebook dùng dữ liệu thật đã có trong `study/kuairec_data/KuaiRec 2.0/data`.
Không tải dữ liệu, không cài thư viện, không sửa CSV gốc.

**Mục tiêu:** hiểu một dòng đại diện cho gì; đánh giá chất lượng dữ liệu; tìm phân bố hành vi,
độ phổ biến, thời gian và nội dung; từ đó chọn nhãn và feature cho content-based/hybrid.

**Cách chạy bằng VS Code**
1. Mở thư mục `C:/Users/Legion/Downloads/ThucHanh`, mở notebook trong `study/EDA`.
2. Chọn **Select Kernel → Python Environments → .venv**.
   Nếu chưa thấy, chọn interpreter `C:/Users/Legion/Downloads/ThucHanh/.venv/Scripts/python.exe`.
3. Chạy ô kiểm tra môi trường bên dưới, rồi chạy từng ô. Phần quét toàn bộ CSV có thể mất vài phút.
4. Nếu VS Code chưa có hỗ trợ notebook, cần extension Python và Jupyter của Microsoft.
   Notebook không yêu cầu JupyterLab hoặc chạy Docker.

**Nguồn:** [KuaiRec và schema](https://kuairec.com/),
[bài báo](https://arxiv.org/pdf/2202.10842),
[repository](https://github.com/chongminggao/KuaiRec).
Các kết quả bên dưới được tính từ CSV của bạn, không lấy số công bố thay cho kết quả.


In [ ]:
from pathlib import Path
import sys
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print('Python:', sys.executable)
print('NumPy:', np.__version__, '| pandas:', pd.__version__)
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.2})


## 1. Xác định file và phạm vi phân tích

`small_matrix`/`big_matrix`: một dòng là bản ghi phản hồi người dùng–video.
Đừng mặc định một dòng luôn là một lần phát độc lập: hãy kiểm tra cặp trùng và đọc định nghĩa nhãn.
`item_daily_features`: một dòng là video–ngày. `user_features`, categories và caption là metadata.

Ban đầu đọc **100.000 dòng đầu của small_matrix** để học pandas nhanh.
Đây là phần đầu file, **không phải mẫu ngẫu nhiên đại diện toàn bộ tập**.
`df.sample()` sau đó chỉ lấy ngẫu nhiên trong phần đầu này.


In [ ]:
STUDY_DIR = Path(r'C:/Users/Legion/Downloads/ThucHanh/study')
DATA_DIR = STUDY_DIR / 'kuairec_data' / 'KuaiRec 2.0' / 'data'
N_PREVIEW = 100_000
CHUNK_SIZE = 250_000
RUN_FULL_SCAN = True       # False nếu hôm nay chỉ học phần cơ bản
RUN_PAIR_AUDIT = False     # True: đọc cặp user-video đầy đủ để kiểm tra density/overlap

assert DATA_DIR.exists(), f'Không tìm thấy dữ liệu: {DATA_DIR}'
files = pd.DataFrame([
    {'file': p.name, 'size_MB': p.stat().st_size / 1_000_000}
    for p in sorted(DATA_DIR.glob('*.csv'))
])
display(files.round(2))


In [ ]:
df = pd.read_csv(DATA_DIR / 'small_matrix.csv', nrows=N_PREVIEW)
print('Phạm vi: phần đầu small_matrix, số dòng đọc =', len(df))
display(df.head())
display(df.tail())
display(df.sample(5, random_state=42))


## 2. Những lệnh pandas nên biết

| Lệnh | Câu hỏi |
|---|---|
| `head`, `tail`, `sample` | Dữ liệu trông như thế nào? |
| `shape`, `columns`, `dtypes`, `info` | Có bao nhiêu dòng/cột, kiểu gì, tốn RAM bao nhiêu? |
| `describe`, `quantile` | Giá trị thường gặp và đuôi phân bố ra sao? |
| `isna`, `duplicated`, `nunique` | Thiếu, trùng và độ đa dạng? |
| `value_counts`, `groupby`, `agg` | Nhóm nào phổ biến hoặc có hành vi khác? |
| `loc`, `query`, `sort_values` | Lọc và tìm bản ghi cần kiểm tra? |
| `merge`, `explode`, `pivot_table` | Nối metadata, mở danh sách tag, so sánh nhóm? |
| `pd.to_datetime`, `pd.cut` | Phân tích thời gian và chia khoảng số? |

NumPy giúp tính vector, phân vị, điều kiện và histogram. Phần lớn EDA dạng bảng dùng pandas.
ID là **định danh**, trung bình `user_id` không mang ý nghĩa sở thích.


In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.info(memory_usage='deep')
display(df.dtypes.rename('dtype').to_frame())
display(df[['play_duration', 'video_duration', 'watch_ratio']].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]
))
display(df.select_dtypes(include=['object', 'string']).describe())


## 3. Chất lượng dữ liệu trước khi suy luận

Kiểm tra dữ liệu thiếu, số âm, video dài 0, giá trị không hữu hạn và tính nhất quán công thức.
`watch_ratio > 1` có thể do xem lại, không tự động là lỗi.
Cặp user–video trùng cũng chưa đủ để kết luận phải xóa: xem toàn bộ bản ghi trước.
Kết quả ở đây chỉ áp dụng cho phần đầu file.


In [ ]:
quality = pd.DataFrame({
    'missing': df.isna().sum(),
    'missing_pct': df.isna().mean() * 100,
    'nunique': df.nunique(dropna=True)
})
display(quality)
print('Dòng trùng hoàn toàn:', df.duplicated().sum())
print('Cặp user-video lặp (số dòng ngoài lần xuất hiện đầu):',
      df.duplicated(['user_id', 'video_id']).sum())

valid = (df.video_duration > 0) & df.play_duration.notna() & np.isfinite(df.watch_ratio)
bad_formula = ~np.isclose(
    df.loc[valid, 'watch_ratio'],
    df.loc[valid, 'play_duration'] / df.loc[valid, 'video_duration'],
    rtol=1e-5, atol=1e-6
)
display(pd.Series({
    'duration_zero_or_negative': (df.video_duration <= 0).sum(),
    'play_negative': (df.play_duration < 0).sum(),
    'watch_ratio_nonfinite': (~np.isfinite(df.watch_ratio)).sum(),
    'ratio_formula_mismatch_on_valid_rows': bad_formula.sum()
}).rename('count').to_frame())
display(df.nlargest(10, 'watch_ratio'))


## 4. Nhãn phản hồi: xem nhiều có phải thích không?

`watch_ratio = play_duration / video_duration`. Bài báo dùng `watch_ratio > 2`
làm nhãn quan tâm mạnh. Đây là **proxy**, không phải lượt bấm like thật.
Thử ngưỡng 0.5, 1, 2 để thấy việc chọn nhãn thay đổi tỷ lệ positive thế nào.
Không coi cặp chưa quan sát là negative. Nếu thay nhãn, phải đánh giá lại mô hình.


In [ ]:
wr = df.loc[np.isfinite(df.watch_ratio) & (df.watch_ratio >= 0), 'watch_ratio']
display(pd.DataFrame([
    {'threshold': t, 'rule': f'watch_ratio > {t}',
     'positive_rows': int((wr > t).sum()), 'positive_pct': (wr > t).mean() * 100}
    for t in [0.5, 1.0, 2.0]
]))
print('Phân vị NumPy:', dict(zip(['p50', 'p90', 'p99'], np.quantile(wr, [0.5, 0.9, 0.99]))))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
wr.clip(upper=5).hist(bins=50, ax=axes[0])
axes[0].set(title='Preview: watch ratio (values >5 pooled at 5)', xlabel='watch_ratio', ylabel='Records')
np.log1p(wr).hist(bins=50, ax=axes[1])
axes[1].set(title='Preview: log(1 + watch ratio)', xlabel='log1p(watch_ratio)', ylabel='Records')
plt.tight_layout()
plt.show()


## 5. Độ dài video và hành vi xem

Video ngắn dễ có ratio cao hơn? So sánh median và tỷ lệ positive theo độ dài.
Đây là liên hệ quan sát, chưa chứng minh độ dài gây ra hành vi.
`corr` chỉ đo quan hệ tuyến tính; ID không nên đưa vào bảng correlation.


In [ ]:
df['video_seconds'] = df.video_duration / 1000
df['duration_band'] = pd.cut(df.video_seconds,
    bins=[0, 7, 15, 30, 60, np.inf], labels=['0-7s', '7-15s', '15-30s', '30-60s', '>60s'])
df['strong_interest'] = df.watch_ratio.gt(2).where(df.watch_ratio.notna())
duration_summary = df.groupby('duration_band', observed=False).agg(
    records=('watch_ratio', 'size'), median_ratio=('watch_ratio', 'median'),
    strong_interest_rate=('strong_interest', 'mean'))
display(duration_summary)
display(df[['play_duration', 'video_duration', 'watch_ratio']].corr())
duration_summary.strong_interest_rate.plot.bar(title='Preview: strong-interest rate by duration')
plt.ylabel('Fraction with watch_ratio > 2')
plt.tight_layout()
plt.show()


## 6. Người dùng, độ phổ biến và thời gian trong preview

`groupby` là công cụ trọng tâm: chuyển từ dòng tương tác sang bảng thống kê người dùng/video.
Số bản ghi của video phản ánh exposure và phản hồi, không tự động là chất lượng.
Thời gian lấy từ chuỗi `time` trong CSV. Không tự đổi sang múi giờ máy.


In [ ]:
user_preview = df.groupby('user_id').agg(
    records=('video_id', 'size'), unique_videos=('video_id', 'nunique'),
    median_ratio=('watch_ratio', 'median'))
video_preview = df.groupby('video_id').agg(
    records=('user_id', 'size'), unique_users=('user_id', 'nunique'),
    strong_interest_rate=('strong_interest', 'mean'))
display(user_preview.sort_values('records', ascending=False).head(10))
display(video_preview.sort_values('records', ascending=False).head(10))

df['event_time'] = pd.to_datetime(df['time'], errors='coerce')
print('Thời gian parse lỗi:', df.event_time.isna().sum())
print('Khoảng thời gian preview:', df.event_time.min(), '→', df.event_time.max())
df.groupby(df.event_time.dt.floor('D')).size().plot(title='Preview: records by calendar day')
plt.ylabel('Records')
plt.tight_layout()
plt.show()


## 7. Quét toàn bộ bằng chunks: số liệu chính xác với RAM vừa phải

Không tải toàn bộ file lớn vào một DataFrame. Đọc từng khối và cộng thống kê.
Phần này tính số dòng, unique user/video, missing, histogram, tỷ lệ `>2`, bản ghi theo ngày,
theo user/video và theo khoảng độ dài trên **toàn bộ từng file**.

**Không dùng số dòng làm số cặp unique để tính density.** Audit cặp unique có ô riêng.
Không cộng median/quantile của các chunk để lấy median/quantile toàn tập.
Chạy ô này một lần, các ô sau dùng lại kết quả. `RUN_FULL_SCAN=False` sẽ bỏ qua phần nặng.


In [ ]:
CORE_COLS = ['user_id', 'video_id', 'play_duration', 'video_duration', 'date', 'watch_ratio']
RATIO_BINS = [0, 0.25, 0.5, 1, 2, 5, 10, np.inf]
DURATION_BINS = [0, 7000, 15000, 30000, 60000, np.inf]

def add_counts(old, new):
    return old.add(new, fill_value=0)

def scan_interactions(path):
    result = {'rows': 0, 'user_counts': pd.Series(dtype='float64'),
              'video_counts': pd.Series(dtype='float64'), 'daily_counts': pd.Series(dtype='float64'),
              'positive_by_video': pd.Series(dtype='float64'),
              'valid_by_video': pd.Series(dtype='float64'),
              'missing': pd.Series(0, index=CORE_COLS, dtype='int64'),
              'ratio_hist': np.zeros(len(RATIO_BINS)-1, dtype='int64'),
              'duration_count': np.zeros(len(DURATION_BINS)-1, dtype='int64'),
              'duration_positive': np.zeros(len(DURATION_BINS)-1, dtype='int64'),
              'valid_ratio': 0, 'positive': 0, 'negative_ratio': 0,
              'nonfinite_ratio': 0, 'invalid_duration': 0, 'formula_mismatch': 0,
              'max_ratio': -np.inf}
    for i, chunk in enumerate(pd.read_csv(path, usecols=CORE_COLS, chunksize=CHUNK_SIZE), start=1):
        result['rows'] += len(chunk)
        result['missing'] += chunk.isna().sum()
        for key, col in [('user_counts', 'user_id'), ('video_counts', 'video_id'), ('daily_counts', 'date')]:
            result[key] = add_counts(result[key], chunk[col].value_counts())
        finite = np.isfinite(chunk.watch_ratio)
        usable = finite & chunk.watch_ratio.ge(0)
        positive = usable & chunk.watch_ratio.gt(2)
        result['valid_ratio'] += int(usable.sum())
        result['positive'] += int(positive.sum())
        result['negative_ratio'] += int(chunk.watch_ratio.lt(0).sum())
        result['nonfinite_ratio'] += int((~finite).sum())
        result['invalid_duration'] += int(chunk.video_duration.le(0).sum())
        result['max_ratio'] = max(result['max_ratio'], chunk.loc[usable, 'watch_ratio'].max())
        result['positive_by_video'] = add_counts(result['positive_by_video'], chunk.loc[positive, 'video_id'].value_counts())
        result['valid_by_video'] = add_counts(result['valid_by_video'], chunk.loc[usable, 'video_id'].value_counts())
        result['ratio_hist'] += np.histogram(chunk.loc[usable, 'watch_ratio'], bins=RATIO_BINS)[0]
        # np.histogram includes lower edge; these duration bands use the same convention.
        duration_ok = usable & chunk.video_duration.gt(0)
        result['duration_count'] += np.histogram(chunk.loc[duration_ok, 'video_duration'], bins=DURATION_BINS)[0]
        result['duration_positive'] += np.histogram(chunk.loc[duration_ok & positive, 'video_duration'], bins=DURATION_BINS)[0]
        formula_ok = finite & chunk.video_duration.gt(0) & chunk.play_duration.notna()
        result['formula_mismatch'] += int((~np.isclose(
            chunk.loc[formula_ok, 'watch_ratio'],
            chunk.loc[formula_ok, 'play_duration'] / chunk.loc[formula_ok, 'video_duration'],
            rtol=1e-5, atol=1e-6)).sum())
        if i % 10 == 0:
            print(path.name, ':', f"{result['rows']:,}", 'dòng')
    for key in ['user_counts', 'video_counts', 'daily_counts', 'positive_by_video', 'valid_by_video']:
        result[key] = result[key].astype('int64')
    return result

full = {}
if RUN_FULL_SCAN:
    for name in ['small_matrix', 'big_matrix']:
        full[name] = scan_interactions(DATA_DIR / f'{name}.csv')
        print('Hoàn tất', name)
else:
    print('Đã bỏ qua full scan. Bật RUN_FULL_SCAN rồi chạy lại từ ô này để có thống kê toàn tập.')


In [ ]:
if full:
    overview = pd.DataFrame([{
        'matrix': name, 'records': r['rows'], 'users': len(r['user_counts']),
        'videos': len(r['video_counts']), 'valid_ratio_rows': r['valid_ratio'],
        'strong_interest_pct': 100 * r['positive'] / r['valid_ratio'],
        'max_watch_ratio': r['max_ratio'], 'invalid_duration': r['invalid_duration'],
        'formula_mismatch': r['formula_mismatch']
    } for name, r in full.items()])
    display(overview)
    display(pd.DataFrame({name: r['missing'] for name, r in full.items()}))
    display(pd.DataFrame({name: r['ratio_hist'] / r['valid_ratio'] for name, r in full.items()},
        index=['[0,.25)', '[.25,.5)', '[.5,1)', '[1,2)', '[2,5)', '[5,10)', '[10,infinity]']))
    print('Histogram [2,5) gồm cả đúng 2; nhãn positive dùng >2 nên hai số không đồng nhất.')


## 8. Long tail và lịch sử: so sánh big với small

Small được thiết kế để gần đầy đủ nên exposure theo video có thể khá đồng đều.
Big phù hợp hơn để xem chênh lệch độ phổ biến.
Đường tích lũy cho biết top x% video chiếm bao nhiêu % bản ghi.
Top 10% ở đây tính theo **số video**, không phải 10% dòng.


In [ ]:
if full:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, r in full.items():
        counts = r['video_counts'].sort_values(ascending=False)
        x = np.arange(1, len(counts)+1) / len(counts)
        axes[0].plot(x, counts.cumsum() / counts.sum(), label=name)
        top_n = max(1, int(np.ceil(len(counts) * 0.10)))
        print(name, ': top 10% video chiếm', round(100*counts.iloc[:top_n].sum()/counts.sum(), 2), '% bản ghi')
        daily = r['daily_counts'].sort_index().copy()
        daily.index = pd.to_datetime(daily.index.astype('int64').astype(str), format='%Y%m%d')
        axes[1].plot(daily.index, daily.values, label=name)
    axes[0].set(title='Full data: popularity concentration', xlabel='Fraction of videos (most frequent first)', ylabel='Cumulative fraction of records')
    axes[1].set(title='Full data: records by date', ylabel='Records')
    for ax in axes:
        ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame({name: r['user_counts'].describe() for name, r in full.items()}))
    display(pd.DataFrame({name: np.divide(r['duration_positive'], r['duration_count'],
        out=np.full(5, np.nan), where=r['duration_count'] > 0) for name, r in full.items()},
        index=['(0,7)s', '[7,15)s', '[15,30)s', '[30,60)s', '[60,infinity)s']))


## 9. Audit cặp unique, density và overlap (tùy chọn)

Đọc thêm hai cột ID của mỗi file. Dùng NumPy mã hóa cặp và `unique`, thay vì tạo ma trận dense.
Phần này cần thêm RAM và lượt đọc đĩa. Số dòng / (users × videos) **chỉ là density nếu không có cặp lặp**.
Audit giúp kiểm chứng hai ma trận không trùng cặp; không chứng minh được train xảy ra trước test.


In [ ]:
if RUN_PAIR_AUDIT:
    if not full:
        raise ValueError('Bật RUN_FULL_SCAN và chạy phần 7 trước.')
    stride = int(max(r['video_counts'].index.max() for r in full.values())) + 1
    pairs = {}
    audit = []
    for name, r in full.items():
        parts = []
        for chunk in pd.read_csv(DATA_DIR / f'{name}.csv', usecols=['user_id', 'video_id'], chunksize=CHUNK_SIZE):
            assert chunk.notna().all().all(), 'ID thiếu: cần xử lý trước khi mã hóa cặp'
            u = chunk.user_id.to_numpy(dtype=np.int64)
            v = chunk.video_id.to_numpy(dtype=np.int64)
            assert (u >= 0).all() and (v >= 0).all() and (v < stride).all()
            parts.append(u * stride + v)
        pairs[name] = np.unique(np.concatenate(parts))
        del parts
        n_pairs = len(pairs[name])
        audit.append({'matrix': name, 'unique_pairs': n_pairs,
            'repeat_records': r['rows']-n_pairs,
            'density_pct': 100*n_pairs/(len(r['user_counts'])*len(r['video_counts']))})
    display(pd.DataFrame(audit))
    print('Cặp overlap:', len(np.intersect1d(pairs['small_matrix'], pairs['big_matrix'], assume_unique=True)))
    del pairs
else:
    print('Audit cặp đầy đủ chưa chạy. Không suy density từ số dòng.')


## 10. Metadata cho content-based và hybrid

Đọc category/tag/caption. Chuỗi dạng `[27, 9]` cần parse thành list rồi `explode`.
Dùng `ast.literal_eval`, không dùng `eval`. Không nối trực tiếp nhiều bảng one-to-many vào interaction:
một dòng có thể bị nhân lên và làm sai thống kê.
`merge(validate=...)` giúp phát hiện quan hệ nối sai. `UNKNOWN` cũng là thiếu về mặt ngữ nghĩa,
dù `isna()` trả về False.

**Phát hiện ở file caption hiện có:** dấu CR trong văn bản khiến parser mặc định báo buffer overflow,
và `engine='python'` tạo 4 dòng giả. File dùng LF làm kết thúc bản ghi nên đặt
`lineterminator='\n'`. Kiểm tra ID sau đọc, không dùng `on_bad_lines='skip'` để che vấn đề.


In [ ]:
categories = pd.read_csv(DATA_DIR / 'item_categories.csv')
users = pd.read_csv(DATA_DIR / 'user_features.csv')
captions = pd.read_csv(DATA_DIR / 'kuairec_caption_category.csv', lineterminator='\n')
assert captions.video_id.notna().all() and captions.video_id.is_unique
assert pd.api.types.is_integer_dtype(captions.video_id)
for name, table in [('categories', categories), ('users', users), ('captions', captions)]:
    print(name, table.shape)
    display(table.head(3))

categories['tags'] = categories.feat.map(ast.literal_eval)
tag_long = categories[['video_id', 'tags']].explode('tags').dropna(subset=['tags'])
tag_long['tags'] = tag_long.tags.astype('int64')
display(tag_long.tags.value_counts().rename('videos_per_tag').head(15).to_frame())
display(categories.tags.map(len).value_counts().sort_index().rename('videos').to_frame())

caption_missing = pd.DataFrame({
    'null_pct': captions.isna().mean()*100,
    'unknown_or_empty_pct': captions.apply(lambda s:
        s.astype('string').str.strip().isin(['UNKNOWN', '']).mean()*100)
})
display(caption_missing)
display(captions.first_level_category_name.value_counts().head(15))


In [ ]:
enriched = df.merge(categories[['video_id', 'tags']], on='video_id', how='left', validate='many_to_one')
enriched = enriched.merge(captions[['video_id', 'caption', 'first_level_category_name']],
    on='video_id', how='left', validate='many_to_one')
assert len(enriched) == len(df)
print('Preview: tỷ lệ thiếu tag sau join =', enriched.tags.isna().mean())
display(enriched[['user_id', 'video_id', 'watch_ratio', 'tags', 'caption', 'first_level_category_name']].head())

if full:
    stats = pd.DataFrame({
        'records': full['big_matrix']['video_counts'],
        'valid_ratio_records': full['big_matrix']['valid_by_video'],
        'positive_records': full['big_matrix']['positive_by_video']
    }).fillna({'positive_records': 0}).rename_axis('video_id').reset_index()
    content_stats = captions.merge(stats, on='video_id', how='left', validate='one_to_one')
    by_category = content_stats.groupby('first_level_category_name').agg(
        videos=('video_id', 'nunique'), records=('records', 'sum'),
        valid_ratio_records=('valid_ratio_records', 'sum'), positives=('positive_records', 'sum'))
    by_category['positive_rate'] = by_category.positives / by_category.valid_ratio_records.replace(0, np.nan)
    display(by_category.sort_values('records', ascending=False).head(15))
    print('Tỷ lệ theo bản ghi, không phải trung bình tỷ lệ của từng video; đây là EDA, không phải feature train.')


## 11. Xem sở thích của một người: bước đầu cho content-based

Chọn user có nhiều bản ghi nhất trong preview để dễ quan sát. Bạn có thể đổi `USER_ID`.
Một video có nhiều tag được tính vào nhiều nhóm nên tổng số membership có thể lớn hơn số tương tác.
Hồ sơ này chỉ dùng phần preview, chưa đủ đại diện lịch sử đầy đủ của người đó.


In [ ]:
USER_ID = int(df.user_id.value_counts().index[0])
history = enriched.loc[enriched.user_id.eq(USER_ID)].copy()
display(history.sort_values('watch_ratio', ascending=False)[
    ['video_id', 'watch_ratio', 'caption', 'first_level_category_name']].head(10))
memberships = history.explode('tags').dropna(subset=['tags'])
interest_profile = memberships.groupby('tags').agg(
    memberships=('video_id', 'size'), median_ratio=('watch_ratio', 'median'),
    strong_interest_rate=('strong_interest', 'mean'))
display(interest_profile.sort_values('memberships', ascending=False))
interest_profile.memberships.sort_values().plot.barh(figsize=(10, 9), title=f'Preview: tag memberships for user {USER_ID}')
plt.xlabel('Interaction-tag memberships')
plt.tight_layout()
plt.show()


## 12. User metadata, social graph và video–ngày

Social graph chỉ có một phần người dùng có danh sách bạn trong tập, không đủ thay cho toàn bộ follow graph.
Không mặc định `author_id` dùng cùng namespace với `user_id`.
Like/share/follow trong `item_daily_features` là thống kê video–ngày, **không phải nhãn của từng user**.


In [ ]:
display(users[['user_active_degree', 'is_video_author']].value_counts().head(10))
display(users.isna().mean().sort_values(ascending=False).head(10).rename('missing_fraction'))
social = pd.read_csv(DATA_DIR / 'social_network.csv')
social['friends'] = social.friend_list.map(ast.literal_eval)
display(social.head())
print('User có ít nhất một bạn trong file:', social.loc[social.friends.map(len).gt(0), 'user_id'].nunique())
display(social.friends.map(len).describe())

DAILY_COLS = ['video_id', 'date', 'author_id', 'upload_dt', 'show_cnt', 'play_cnt',
    'like_cnt', 'share_cnt', 'follow_cnt', 'collect_cnt']
daily = pd.read_csv(DATA_DIR / 'item_daily_features.csv', usecols=DAILY_COLS)
display(daily.head())
print('Daily shape:', daily.shape)
print('Video-date lặp:', daily.duplicated(['video_id', 'date']).sum())
display(daily.isna().mean().rename('missing_fraction').to_frame())
display(daily[['show_cnt', 'play_cnt', 'like_cnt', 'share_cnt', 'follow_cnt']].describe())


## 13. Feature theo thời gian: tránh đọc tương lai

Ví dụ bên dưới chỉ dùng thống kê **ngày trước ngày tương tác**: nối video + ngày trước.
Không dùng tổng like của cùng ngày để dự đoán một sự kiện vào buổi sáng ngày đó.
Nếu dùng as-of join cho ngày gần nhất, cần sắp xếp đúng và xác định độ trễ dữ liệu.
Metadata được bổ sung năm 2024 có thể không phản ánh đúng thời điểm năm 2020;
phân biệt thí nghiệm sở thích tĩnh với mô phỏng lịch sử nghiêm ngặt.


In [ ]:
events = df[['user_id', 'video_id', 'event_time']].copy()
events['previous_day'] = events.event_time.dt.normalize() - pd.Timedelta(days=1)
previous_daily = daily[['video_id', 'date', 'like_cnt', 'show_cnt']].copy()
previous_daily['previous_day'] = pd.to_datetime(previous_daily['date'].astype('int64').astype(str), format='%Y%m%d')
previous_daily = previous_daily.drop(columns='date').rename(columns={
    'like_cnt': 'previous_day_like_cnt', 'show_cnt': 'previous_day_show_cnt'})
safe_features = events.merge(previous_daily, on=['video_id', 'previous_day'],
    how='left', validate='many_to_one')
display(safe_features.head())
print('Thiếu thống kê ngày trước:', safe_features.previous_day_show_cnt.isna().mean())
print('Giữ NaN nếu thiếu. Không mặc định thiếu = 0 lượt hiển thị.')


## 14. Viết kết luận EDA và làm bài tập

Điền dựa vào kết quả bạn thấy, ghi rõ toàn tập hay preview:

1. Một dòng interaction là gì? Có cặp lặp không (audit đã chạy chưa)?
2. Có cột thiếu, giá trị không hợp lệ hay ratio cực lớn nào cần kiểm tra?
3. Với ngưỡng `>2`, positive chiếm bao nhiêu? Nếu đổi ngưỡng thì sao?
4. Big và small khác nhau thế nào về exposure và độ phổ biến?
5. Ratio và độ dài có liên hệ gì? Cần biến đổi/cap nhãn thế nào?
6. Bao nhiêu video thiếu caption/tag? Content-based sẽ fallback ra sao?
7. Hai người có cùng chủ đề nhưng hành vi khác nhau không? Hybrid giúp gì?
8. Feature nào có thể biết tại thời điểm gợi ý, feature nào có nguy cơ đọc tương lai?

**Bài tập pandas:** đổi `USER_ID`; tìm 10 video có tỷ lệ quan tâm mạnh cao nhất với ít nhất 100
bản ghi hợp lệ; so sánh top theo exposure và theo tỷ lệ phản hồi. Không gọi đây là kết quả mô hình.

**Bước tiếp:** dùng tag/caption làm content baseline, collaborative làm baseline khác, rồi hybrid.
Không dùng nhãn của tập test hoặc hồ sơ tạo từ toàn bộ lịch sử để đánh giá dự đoán quá khứ.


In [ ]:
# Bài tập có gợi ý; thử tự viết trước rồi chạy.
if full:
    r = full['big_matrix']
    video_eda = pd.DataFrame({'exposures': r['video_counts'],
        'valid_records': r['valid_by_video'], 'positive_records': r['positive_by_video']})
    video_eda['positive_records'] = video_eda.positive_records.fillna(0)
    video_eda['positive_rate'] = video_eda.positive_records / video_eda.valid_records.replace(0, np.nan)
    display(video_eda.query('valid_records >= 100').sort_values('positive_rate', ascending=False).head(10))


## Phát hiện đã kiểm chứng trên dữ liệu của bạn

- Small: 4.676.570 dòng và cặp unique; density 99,620%; nhãn `watch_ratio > 2` chiếm 4,64%.
- Big: 12.530.806 dòng, 10.300.969 cặp unique. Có 2.229.837 dòng ngoài lần xuất hiện đầu của cặp.
  Density theo unique pairs là 13,381%. Con số 16,3% công bố tương ứng mật độ theo số bản ghi;
  cần nêu rõ định nghĩa khi so sánh. Không tự xóa các lượt lặp: có thể là tương tác ở thời điểm khác.
- Hai ma trận không có cặp user–video giao nhau (audit toàn bộ).
- Small có 181.992 dòng thiếu `date` (3,89%). Các ví dụ kiểm tra cũng thiếu `time` và `timestamp`.
  Biểu đồ ngày loại những dòng thiếu ngày, nên không bao phủ toàn bộ tương tác.
  Không tự gán ngày để làm temporal split. Muốn biết số thiếu `time`/`timestamp` toàn tập, quét thêm hai cột đó.
- Có 1.209 dòng ở small và 81 dòng ở big lệch công thức ratio tại `rtol=1e-5`, `atol=1e-6`.
  Kiểm tra độ lệch tuyệt đối/tương đối và tài liệu trước khi quyết định dùng nhãn có sẵn hay tính lại.
- Top 10% video theo exposure chiếm 40,72% bản ghi big, so với 10,05% ở small.
- Caption cần `lineterminator='\n'` để giữ đúng 10.728 video. Đổi sang Python parser đơn thuần
  tạo 4 dòng giả; kiểm tra ID giúp nhận ra vấn đề.

Các số này là kết quả EDA, chưa phải kết quả chất lượng mô hình.
